In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Sparsely-Gated Mixture of Experts (GMoE) in Psyre

This notebook illustrates the trainable sparsely-gated Mixture of Experts from
`examples/gmoe.py`, covering both:

1. **Standalone scalar GMoE**: top-2 sparse softmax gating over 3 quadratic/affine
   experts executing strictly on dynamic subclocks `. on c_k` (`c_k = (g_k != 0.0)`),
   with forward pass, reverse-mode AD (`when`/`merge` adjoints), and a two-step
   SGD training run showing that unselected experts stay `Absent` (`□`) and frozen.
2. **Chained GMoE-RNN**: 3 dynamically dispatched `examples/rnn.py` RNN + ReLU
   perceptron experts unrolled to heterogeneous depths `(3, 4, 5)` cells, with
   pass-constant sparse gating, forward pass, and a two-step multi-pass BPTT
   training run.

In [ ]:
# @title Psyre imports

import logging
# !pip install -q psyre

# pylint: disable=g-importing-member,g-multiple-import
from psyre.analysis import check_causality
from psyre.builder import build_graph
from psyre.clock import check_clocks
from psyre.common import (
    Absent,
    counter,
    logger,
    switch,
)
from examples.gmoe import (
    DEFAULT_EXPERT_CELLS,
    GateParams,
    GmoeParams,
    ScalarExpertParams,
    chained_gmoe_forward,
    chained_param_streams,
    eval_chained_gmoe,
    extract_chained_params,
    extract_scalar_params,
    gmoe_forward,
    train_chained_gmoe,
    train_gmoe,
)
from examples.rnn import (
    DIMENSION,
    ModelParams,
    PerceptronParams,
    RnnParams,
    dataset_streams,
)
from psyre.scheduler import Scheduler
from psyre.stream import Stream
from psyre.trace import (
    is_absent,
    present_values,
)
from psyre.visualize import visualize_graph

# pylint: enable=g-importing-member,g-multiple-import

In [ ]:
# @title Scalar GMoE forward pass (dynamic dispatch & subclocks)

logger.setLevel(logging.WARNING)

scalar_params = GmoeParams(
    gate=GateParams(w=(-1.2, 0.1, 1.3), b=(0.5, -0.5, 0.5)),
    experts=(
        ScalarExpertParams(w1=1.5, w2=-0.2, b=0.8),
        ScalarExpertParams(w1=-0.7, w2=0.15, b=2.5),
        ScalarExpertParams(w1=0.3, w2=0.45, b=-0.4),
    ),
)
xs = [-2.0, 0.0, 2.0]


def scalar_forward_circuit():
  return gmoe_forward(Stream(xs, name='x'), scalar_params.map(Stream))


g_scalar_fwd = build_graph(scalar_forward_circuit)
check_causality(g_scalar_fwd)
clocks_scalar_fwd = check_clocks(g_scalar_fwd)
fwd_out = Scheduler.run_graph(
    g_scalar_fwd, horizon=len(xs), clocks=clocks_scalar_fwd
)

print('Expert output clocks:')
for k in range(3):
  clk = clocks_scalar_fwd.stream_clock(g_scalar_fwd.output.expert_outputs[k])
  print(f'  o{k + 1}: {clk}')

print('\nCycle-by-cycle dynamic dispatch (□ = Absent / skipped):')
for t, xv in enumerate(xs):
  g_str = ', '.join(f'{fwd_out.gates[k][t]:.3f}' for k in range(3))
  o_str = ', '.join(
      '□' if fwd_out.expert_outputs[k][t] is Absent
      else f'{fwd_out.expert_outputs[k][t]:.3f}'
      for k in range(3)
  )
  print(
      f'  t={t} (x={xv:+.1f}): gates=[{g_str}]  '
      f'experts=[{o_str}]  o={fwd_out.o[t]:.4f}'
  )

In [ ]:
# @title Stage-1 schematic: scalar GMoE forward pass

visualize_graph(
    g_scalar_fwd,
    title='Stage-1 SYNC Scalar GMoE Forward Pass',
    height_px=600,
)

In [ ]:
# @title Scalar GMoE two-step training run (max_steps=2)

scalar_dataset = [(-2.0, -3.0), (0.0, 2.2), (2.0, 5.2)]
scalar_steps = 2

scalar_train = Scheduler.run(
    train_gmoe,
    scalar_dataset,
    scalar_params,
    max_steps=scalar_steps,
    lr=0.05,
    horizon=scalar_steps,
)
g_scalar_train = Scheduler.graph

print('Scalar GMoE 2-step training losses:', present_values(scalar_train.loss))
for t in range(scalar_steps):
  prev_p = (
      scalar_params if t == 0 else extract_scalar_params(scalar_train.params, t - 1)
  )
  next_p = extract_scalar_params(scalar_train.params, t)
  skipped = [k for k in range(3) if not scalar_train.conds[k][t]]
  frozen = all(
      is_absent(scalar_train.expert_outputs[k], t)
      and next_p.experts[k] == prev_p.experts[k]
      for k in skipped
  )
  print(
      f'  step {t} (x={scalar_dataset[t][0]:+.1f}): '
      f'loss={scalar_train.loss[t]:.4f}, '
      f'skipped_expert={skipped}, frozen={frozen}'
  )

In [ ]:
# @title Stage-1 schematic: scalar GMoE two-step training

visualize_graph(
    g_scalar_train,
    title='Stage-1 SYNC Scalar GMoE Training (max_steps=2)',
    height_px=640,
)

In [ ]:
# @title Chained GMoE-RNN forward pass (expert unrolling depths (3, 4, 5))

chained_params = GmoeParams(
    gate=GateParams(
        w=(
            [1.2, -0.4, 0.3, -0.2, 0.1],
            [-0.3, 1.1, -0.5, 0.4, -0.2],
            [-0.8, -0.5, 1.3, 0.3, 0.6],
        ),
        b=(0.15, -0.05, 0.1),
    ),
    experts=(
        ModelParams(
            rnn=RnnParams(wxh=0.35, whh=-0.25, bh=0.1),
            per=PerceptronParams(
                wxh=[0.3, 0.4, -0.2, 0.5, 0.35],
                bh=[0.6, 0.0, 0.0, 0.0, 0.0],
            ),
        ),
        ModelParams(
            rnn=RnnParams(wxh=-0.3, whh=0.4, bh=0.15),
            per=PerceptronParams(
                wxh=[0.45, -0.25, 0.35, 0.2, 0.5],
                bh=[0.7, 0.0, 0.0, 0.0, 0.0],
            ),
        ),
        ModelParams(
            rnn=RnnParams(wxh=0.25, whh=0.3, bh=-0.1),
            per=PerceptronParams(
                wxh=[0.2, 0.35, 0.45, -0.15, 0.4],
                bh=[0.8, 0.0, 0.0, 0.0, 0.0],
            ),
        ),
    ),
)

chained_dataset = [
    ([1.0, 0.2, -0.8, 0.1, -0.3], 2.0),
    ([-0.9, 0.8, 0.9, 0.5, 0.6], 1.5),
    ([0.8, -0.9, 0.9, -0.2, 0.4], 1.8),
]


def chained_forward_circuit():
  tick = counter()
  pass_idx = tick // DIMENSION
  in_pass = tick % DIMENSION
  pass_reset = in_pass == 0
  samples = dataset_streams(chained_dataset, in_pass=in_pass)
  x = switch(
      pass_idx, {i: s.x for i, s in enumerate(samples)}, default=0.0
  )
  p_streams = chained_param_streams(in_pass, chained_params)
  return chained_gmoe_forward(
      x,
      p_streams,
      expert_cells=DEFAULT_EXPERT_CELLS,
      in_pass=in_pass,
      pass_reset=pass_reset,
  )


g_chained_fwd = build_graph(chained_forward_circuit)
check_causality(g_chained_fwd)
clocks_chained_fwd = check_clocks(g_chained_fwd)
chained_fwd = Scheduler.run_graph(
    g_chained_fwd,
    horizon=len(chained_dataset) * DIMENSION,
    clocks=clocks_chained_fwd,
)

print(f'Expert RNN unrolling depths: {DEFAULT_EXPERT_CELLS}')
for p_idx in range(len(chained_dataset)):
  start, end = p_idx * DIMENSION, (p_idx + 1) * DIMENSION
  g_vals = [chained_fwd.gates[k][end - 1] for k in range(3)]
  skipped = [k for k in range(3) if not chained_fwd.conds[k][start]]
  all_absent = all(
      chained_fwd.expert_outputs[k][start:end] == [Absent] * DIMENSION
      for k in skipped
  )
  print(
      f'  pass {p_idx}: gates={[round(g, 3) for g in g_vals]}, '
      f'skipped_expert={skipped} (all Absent={all_absent}), '
      f'o[-1]={chained_fwd.o[end - 1]:.4f}'
  )

In [ ]:
# @title Stage-1 schematic: chained GMoE-RNN forward pass (cells=(3, 4, 5))

visualize_graph(
    g_chained_fwd,
    title='Stage-1 SYNC Chained GMoE-RNN Forward Pass (cells=(3, 4, 5))',
    height_px=660,
)

In [ ]:
# @title Chained GMoE-RNN two-step BPTT training run (cells=(3, 4, 5), max_steps=2)

chained_steps = 2
# 1 INIT_LOAD pass + 2 BPTT passes = 3 passes (15 ticks).
train_horizon = (1 + chained_steps) * DIMENSION

chained_train = Scheduler.run(
    train_chained_gmoe,
    chained_dataset,
    chained_params,
    expert_cells=DEFAULT_EXPERT_CELLS,
    max_steps=chained_steps,
    lr=0.05,
    horizon=train_horizon,
)
g_chained_train = Scheduler.graph

train_losses = [float(v) for v in present_values(chained_train.loss)]
print('Chained GMoE-RNN 2-step training losses:', train_losses)

for step in range(chained_steps):
  pass_idx = step + 1
  start, end = pass_idx * DIMENSION, (pass_idx + 1) * DIMENSION
  prev_p = extract_chained_params(
      chained_train.params.map(lambda seq: seq[(pass_idx - 1) * DIMENSION:start])
  )
  next_p = extract_chained_params(
      chained_train.params.map(lambda seq: seq[start:end])
  )
  skipped = [k for k in range(3) if not chained_train.conds[k][start]]
  frozen = all(
      chained_train.expert_outputs[k][start:end] == [Absent] * DIMENSION
      and next_p.experts[k] == prev_p.experts[k]
      for k in skipped
  )
  print(
      f'  step {step}: loss={train_losses[step]:.4f}, '
      f'skipped_expert={skipped}, frozen={frozen}'
  )

trained_chained = extract_chained_params(chained_train.params)
eval_horizon = len(chained_dataset) * DIMENSION
init_eval = present_values(
    Scheduler.run(
        eval_chained_gmoe,
        chained_dataset,
        chained_params,
        expert_cells=DEFAULT_EXPERT_CELLS,
        horizon=eval_horizon,
    )
)
trained_eval = present_values(
    Scheduler.run(
        eval_chained_gmoe,
        chained_dataset,
        trained_chained,
        expert_cells=DEFAULT_EXPERT_CELLS,
        horizon=eval_horizon,
    )
)
print(
    f'Dataset eval loss: initial={sum(init_eval):.4f} '
    f'-> after 2 steps={sum(trained_eval):.4f}'
)

In [ ]:
# @title Stage-1 schematic: chained GMoE-RNN two-step BPTT training

visualize_graph(
    g_chained_train,
    title='Stage-1 SYNC Chained GMoE-RNN Training (cells=(3, 4, 5), max_steps=2)',
    height_px=680,
)